# Build a custom filter with Dynestyx

A custom filter plugs into the same `dsx.Filter` handler as a built-in filter. You supply two callbacks. `build_filter(model)` binds an algorithm to the current model, and `gather_outputs` selects what each observation contributes to the result. Dynestyx runs the time loop, constructs posterior distributions, and registers the likelihood when you call `dsx.sample`.

We'll implement the same Gauss–Hermite algorithm twice: once with split prediction and update callbacks, and once with Cuthbert’s combined callback. Both versions expose the innovation as a custom diagnostic. We check them against each other and an exact Kalman filter. The final example compares the custom filter with Dynestyx's built-in UKF on a transition with an interaction between state coordinates.

In [ ]:
from typing import NamedTuple

import jax
import jax.numpy as jnp
import numpy as np
import numpyro.distributions as dist
from numpyro.handlers import seed, trace
from cuthbert.inference import Filter as CuthbertFilter
from cuthbertlib.quadrature import gauss_hermite

import dynestyx as dsx
from dynestyx.configs import CustomFilterConfig, KFConfig, UKFConfig
from dynestyx.inference.custom_filter import FilterOutputs, SplitFilter
from dynestyx.inference.observation_predictions import PredictedObservationOutputs

## Decide what a step returns

The state is yours to design. This algorithm needs a filtered mean, covariance, and running log likelihood for the next step. It also carries the predictive observation moments and innovation so the gather callback can expose them. Keep the same field shapes at every step for JAX's scan.

In [ ]:
class Prior(NamedTuple):
    mean: jax.Array
    cov: jax.Array
    loglik: jax.Array  # Evidence accumulated through the previous observation.


class State(NamedTuple):
    mean: jax.Array
    cov: jax.Array
    loglik: jax.Array
    predictions: PredictedObservationOutputs
    innovation: jax.Array

In [ ]:
def gather_outputs(state):
    return FilterOutputs(
        log_normalizing_constant=state.loglik,
        filtered_means=state.mean,
        filtered_covariances=state.cov,
        predicted_observations=state.predictions,
        extra_outputs={"innovation": state.innovation},
    )

`FilterOutputs` is the boundary between your algorithm and Dynestyx. The Gaussian fields become one distribution per observation in `result.dists`. The last cumulative log likelihood becomes `result.marginal_loglik`, and named extras are stacked along observation time. Here the innovation is $y_k-\hat y_{k|k-1}$, so saving it also gives us a way to check that predictions were gathered before assimilation.

## Implement the split interface

The factory binds the model and quadrature rule once. At the first observation Dynestyx calls `initialize` and passes its prior straight to `update`. At later observations it calls `predict` and then `update`. The two-argument gather callback receives both the prior and the filtered state. Here the update already saved the innovation, so gathering needs only the filtered state.

In [ ]:
def build_split_filter(model):
    rule = gauss_hermite.weights(model.state_dim, order=3)

    def initialize(inputs, *, key):
        return initial_prior(model)

    def predict(filtered, inputs, *, key):
        return predict_prior(model, rule, filtered, inputs)

    def update(prior, inputs, *, key):
        return update_state(model, rule, prior, inputs)

    return SplitFilter(initialize=initialize, predict=predict, update=update)


def gather_split(prior, filtered):
    return gather_outputs(filtered)


split_config = CustomFilterConfig(build_filter=build_split_filter, gather_outputs=gather_split)

## Implement the combined interface

Cuthbert separates preparation from combination. `prepare` can package the current inputs before the previous belief is available. Here it just forwards them. `combine` owns the first-step decision and returns a complete filtered state. Its gather callback takes that state as its sole argument. Both factories use the same numerical helpers.

In [ ]:
def build_combined_filter(model):
    rule = gauss_hermite.weights(model.state_dim, order=3)

    def initialize(*, key):
        prior = initial_prior(model)
        # The initial scan carry has the same structure as every filtered state.
        zero = jnp.zeros(model.observation_dim, dtype=prior.mean.dtype)
        zeros = jnp.outer(zero, zero)
        predictions = PredictedObservationOutputs(mean=zero, cov=zeros, obs_cov=zeros)
        return State(prior.mean, prior.cov, prior.loglik, predictions, zero)

    def prepare(inputs, *, key):
        return inputs

    def combine(filtered, inputs):
        prior = jax.lax.cond(
            inputs.is_first_step,
            lambda: Prior(filtered.mean, filtered.cov, filtered.loglik),
            lambda: predict_prior(model, rule, filtered, inputs),
        )
        return update_state(model, rule, prior, inputs)

    return CuthbertFilter(init_prepare=initialize, filter_prepare=prepare,
                          filter_combine=combine, associative=False)


combined_config = CustomFilterConfig(build_filter=build_combined_filter,
                                     gather_outputs=gather_outputs)

## Share the numerical steps

This example uses order-three Gauss–Hermite quadrature. For a Gaussian input, `transform` evaluates the model at weighted points and returns four quantities: the output mean, the covariance of the signal, the average noise covariance, and the state–signal cross-covariance. `update_state` uses them for a Gaussian update and adds the predictive log density of this observation to the running total.

In [ ]:
def transform(rule, mean, cov, conditional):
    points = rule.get_sigma_points(mean, jnp.linalg.cholesky(cov))

    def moments(x):
        distribution = conditional(x)
        return distribution.mean, distribution.covariance_matrix

    means, covs = jax.vmap(moments)(points.points)
    signals = points._replace(points=means)
    noise = jnp.einsum("i,ijk->jk", points.wm, covs)
    return signals.mean, signals.covariance(), noise, points.covariance(signals)


def initial_prior(model):
    initial = model.initial_condition
    return Prior(initial.mean, initial.covariance_matrix, jnp.zeros_like(initial.mean[0]))


def predict_prior(model, rule, filtered, inputs):
    def transition(x):
        return model.state_evolution(x, inputs.u_prev, inputs.time_prev, inputs.time)

    mean, signal, noise, _ = transform(rule, filtered.mean, filtered.cov, transition)
    return Prior(mean, signal + noise, filtered.loglik)


def update_state(model, rule, prior, inputs):
    mean, cov, previous_loglik = prior
    y = inputs.y

    def observe(x):
        return model.observation_model(x, inputs.u, inputs.time)

    y_mean, signal, noise, cross = transform(rule, mean, cov, observe)
    y_cov = signal + noise
    gain = jnp.linalg.solve(y_cov, cross.T).T
    innovation = y - y_mean
    return State(
        mean=mean + gain @ innovation,
        cov=cov - gain @ cross.T,
        loglik=previous_loglik + dist.MultivariateNormal(y_mean, y_cov).log_prob(y),
        predictions=PredictedObservationOutputs(mean=y_mean, cov=signal, obs_cov=y_cov),
        innovation=innovation,
    )

## Use both configs with conditioning and NumPyro

Consider two independent standard-normal coordinates. We observe the first with noise variance $0.5$. In the transition, the product of both coordinates becomes the next first coordinate, with process noise variance $0.1$. Both filters below receive the same model and the observations 0 and 1.

In [ ]:
q, r = 0.1, 0.5
model = dsx.DynamicalModel(
    control_dim=0,
    initial_condition=dist.MultivariateNormal(jnp.zeros(2), jnp.eye(2)),
    state_evolution=dsx.GaussianStateEvolution(
        F=lambda x, u, t, t_next: jnp.array([x[0] * x[1], x[1]]),
        cov=q * jnp.eye(2),
    ),
    observation_model=dsx.LinearGaussianObservation(H=jnp.array([[1., 0.]]),
                                                    R=r * jnp.eye(1)),
)
data = dict(obs_times=jnp.array([0., 1.]), obs_values=jnp.array([[0.], [1.]]))
configs = {"Split": split_config, "Combined": combined_config}
results = {}
for name, config in configs.items():
    with dsx.Filter(config):
        results[name] = dsx.condition("x", model, **data)

split, combined = results["Split"], results["Combined"]
# Compare the entire gathered history, including covariances, diagnostics, and evidence.
assert jax.tree.structure(split.states) == jax.tree.structure(combined.states)
for left, right in zip(jax.tree.leaves(split.states), jax.tree.leaves(combined.states), strict=True):
    np.testing.assert_allclose(left, right, atol=1e-6)
np.testing.assert_allclose(split.extra_outputs["innovation"],
                           data["obs_values"] - split.predicted_observations.mean, atol=1e-6)
for name, result in results.items():
    print(name, "filtered mean after the second observation:", result.dists[-1].mean)

In [ ]:
for name, config in configs.items():
    with trace() as sites, seed(rng_seed=0), dsx.Filter(config):
        dsx.sample("x", model, **data)
    factor = sites["x_marginal_log_likelihood"]["fn"].log_factor
    np.testing.assert_allclose(factor, results[name].marginal_loglik, atol=1e-6)
    print(name, "log likelihood registered with NumPyro:", factor)

## Check the callback wiring on a linear model

Each factory can bind to a different model. On linear dynamics and observations, the quadrature moments are exact. Comparing every posterior and every cumulative likelihood with the Kalman filter checks initialization, time alignment, and the output mapping.

In [ ]:
linear = dsx.LTI_discrete(A=jnp.array([[0.8]]), Q=0.1*jnp.eye(1),
                          H=jnp.eye(1), R=0.2*jnp.eye(1))
with dsx.Filter(KFConfig(filter_source="cuthbert")):
    exact = dsx.condition("x", linear, **data)
for config in configs.values():
    with dsx.Filter(config):
        gh = dsx.condition("x", linear, **data)
    for a, b in zip(gh.dists, exact.dists, strict=True):
        np.testing.assert_allclose(a.mean, b.mean, atol=1e-5)
        np.testing.assert_allclose(a.covariance_matrix, b.covariance_matrix, atol=1e-5)
    np.testing.assert_allclose(gh.log_normalizing_constant, exact.log_normalizing_constant, atol=1e-5)
    np.testing.assert_allclose(gh.marginal_loglik, exact.marginal_loglik, atol=1e-5)

## Compare with the built-in UKF

After the first observation, the two coordinates are independent with means zero and variances $v=r/(1+r)$ and $1$. Their product has variance $v$, so the exact variance of the second **observation** is $v+q+r$. The UKF's five sigma points in two dimensions lie on the coordinate axes of this diagonal Gaussian. Every point therefore has product zero. The custom rule evaluates nine points, including off-axis points that capture the missing variance.

We can also check the final **filtered mean** against a numerical Bayesian reference. Conditional on the previous second coordinate $b$, the next first coordinate is Gaussian with variance $s_b^2=v b^2+q$. For the second observation $y$, its conditional posterior mean is $y s_b^2/(s_b^2+r)$. Average this over $b$, weighting its standard-normal prior by the observation likelihood $\mathcal N(y;0,s_b^2+r)$.

In [ ]:
from scipy.integrate import quad
from scipy.stats import norm

with dsx.Filter(UKFConfig()):
    ukf = dsx.condition("x", model, **data)

v = r / (1 + r)
y = float(data["obs_values"][-1, 0])

def state_variance(b):
    return v * b**2 + q

def weight(b):
    return norm.pdf(b) * norm.pdf(y, scale=np.sqrt(state_variance(b) + r))

def weighted_mean(b):
    variance = state_variance(b)
    return weight(b) * y * variance / (variance + r)

evidence = quad(weight, -np.inf, np.inf)[0]
reference_mean = quad(weighted_mean, -np.inf, np.inf)[0] / evidence
reference_var = v + q + r

# The discrete UKF stores the state forecast from observation 0 to observation 1.
# H selects coordinate 0, so adding R gives its predictive observation variance.
rows = [
    ("Bayesian reference", reference_var, reference_mean),
    ("Split Gauss–Hermite", float(split.predicted_observations.obs_cov[1, 0, 0]),
     float(split.dists[1].mean[0])),
    ("Combined Gauss–Hermite", float(combined.predicted_observations.obs_cov[1, 0, 0]),
     float(combined.dists[1].mean[0])),
    ("Built-in UKF", float(ukf.states.predicted_covariances[0, 0, 0]) + r,
     float(ukf.dists[1].mean[0])),
]
print(f"{'Method':<24} {'Observation variance':>22} {'Filtered mean':>16} {'Abs. mean error':>17}")
for name, variance, mean in rows:
    print(f"{name:<24} {variance:>22.4f} {mean:>16.4f} {abs(mean-reference_mean):>17.4f}")
np.testing.assert_allclose(rows[1][1], reference_var, atol=1e-5)

The custom filter gives the exact predictive observation variance of **0.9333**. The UKF gives **0.6000**, so it gives the second observation less weight. Against the numerical Bayesian reference, the absolute error in the filtered mean is **0.0831** for the custom filter and **0.2145** for the UKF. The remaining error comes from the Gaussian approximation to the product distribution. The custom rule uses $3^d$ model evaluations per transform, compared with $2d+1$ for the UKF.